# Synthetic Factory Load Profile Sanity Check (Issue #7)

This notebook verifies the synthetic 15-minute electricity load profile generated for the RMG factory in Ashulia, Dhaka.
It inspects:
1. Weekly schedule and Friday weekend drop.
2. Comparison of a hot summer day vs a mild winter day.
3. Separation between inflexible base machinery load and flexible HVAC cooling load.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from energy_brain.config import load_site_config

cfg = load_site_config()
load_df = pd.read_parquet("../data/processed/load.parquet")
load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
load_df.set_index("timestamp", inplace=True)

print(f"Total 15-minute slots: {len(load_df):,}")
print(f"Peak Total Load: {load_df['total_kw'].max():.2f} kW (Config Peak: {cfg.load.peak_kw} kW)")
print(f"Standby Base Load: {load_df['base_kw'].min():.2f} kW (Config Base: {cfg.load.base_kw} kW)")
load_df.head()

## 1. Typical Working Week Profile (Saturday to Friday)
Shows active factory shifts (08:00–17:00 + overtime to 20:00) Saturday through Thursday, and the sharp drop on Friday (weekly holiday).

In [ ]:
sample_week = load_df.loc["2024-06-01":"2024-06-07"]
plt.figure(figsize=(14, 5))
plt.plot(sample_week.index, sample_week["total_kw"], label="Total Site Load", color="black", lw=1.5)
plt.plot(sample_week.index, sample_week["base_kw"], label="Inflexible Base Load", color="tab:blue", lw=1.2)
plt.plot(sample_week.index, sample_week["hvac_kw"], label="Flexible HVAC Load", color="tab:cyan", lw=1.2)
plt.title("1-Week Factory Load Profile (Ashulia RMG Factory - June 2024)")
plt.xlabel("Date / Time (Asia/Dhaka)")
plt.ylabel("Power (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Hot Summer Day vs Mild Winter Day
Compares HVAC thermal demand on a hot day (June 3) vs a mild winter day (January 15).

In [ ]:
hot_day = load_df.loc["2024-06-03"]
cool_day = load_df.loc["2024-01-15"]

plt.figure(figsize=(12, 4))
plt.plot(
    hot_day.index.strftime("%H:%M"),
    hot_day["hvac_kw"],
    label="Hot Day HVAC (June 3)",
    color="red",
    lw=2,
)
plt.plot(
    cool_day.index.strftime("%H:%M"),
    cool_day["hvac_kw"],
    label="Cool Day HVAC (Jan 15)",
    color="blue",
    lw=2,
)
plt.xticks(hot_day.index.strftime("%H:%M")[::8], rotation=45)
plt.title("HVAC Cooling Demand: Hot Day vs Cool Day")
plt.xlabel("Time of Day")
plt.ylabel("HVAC Power (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()